# Lab 02 - 3: Solutions #

## Binary image classification on the DogsVsCats dataset using a ResNet-18 ##

In [ ]:
import os
import torch
import torchvision

import numpy as np
import pandas as pd
import torch.nn as nn
import torchvision.models as models

from PIL import Image
# from tqdm import tqdm
from pathlib import Path
from tqdm.notebook import tqdm
from collections import OrderedDict
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights
from torch.utils.data import Dataset, DataLoader


# Dataset path.
dataset_path = Path('dogs_vs_cats')

# Hyperparameters.
LR = 1e-4
EPOCH = 2
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
torch.cuda.is_available()

True

* Here you have to decide whether to build a dataset class from scratch, or use one provided by the torchvision library. Please have a look to:
- https://pytorch.org/tutorials/beginner/basics/data_tutorial.html (Go to section "Creating a Custom Dataset for yout files")
- https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html

In [2]:
# Transformations.
data_transform = transforms.Compose([transforms.Resize((224, 224)),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],)])

## Dataset.
train_data = datasets.ImageFolder(
    root=os.path.join(dataset_path, 'train'),
    transform=data_transform
)
test_data = datasets.ImageFolder(
    root=os.path.join(dataset_path, 'test'),
    transform=data_transform
)

## DataLoaders.
train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
print(f"DogsVsCatsDataset - train: {len(train_data)}")
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"DogsVsCatsDataset - test: {len(test_data)}")

# Count the test labels.
d_labels = 0
c_labels = 0
for i in (titer := tqdm(range(len(test_data)))):
    titer.set_description(f"Check test labels")
    _, label = test_data[i]
    if label == 0:
        c_labels += 1
    else:
        d_labels +=1
print(f"DogsVsCatsDataset - n. of dogs in test: {d_labels}")
print(f"DogsVsCatsDataset - n. of cats in test: {c_labels}")

DogsVsCatsDataset - train: 25000
DogsVsCatsDataset - test: 12461


  0%|          | 0/12461 [00:00<?, ?it/s]

DogsVsCatsDataset - n. of dogs in test: 6219
DogsVsCatsDataset - n. of cats in test: 6242


In [ ]:
## Define the model.
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# Change the output layer to output 1 class instead of 1000 classes.
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)

model = model.to(DEVICE)

## Define the loss function.
# Binary Cross Entropy with sigmoid, so no need to use sigmoid in the model.
criterion = nn.BCEWithLogitsLoss()

## Define the optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [6]:
## Training step.
print(f"Start training on {DEVICE} [...]")
model.train()

for e in range(EPOCH):
    e_loss = 0.0

    for i, data in (tepoch := tqdm(enumerate(train_loader), unit="batch", total=len(train_loader))):
        tepoch.set_description(f"Epoch {e}")
        x, y = data[0].to(DEVICE), data[1].to(DEVICE)

        # Training step for the single batch.
        model.zero_grad()
        outputs = model(x)
        y = y.reshape(-1, 1)
        loss = criterion(outputs, y.float())
        # loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        # Print statistics.
        e_loss += loss.item() * x.shape[0]
        tepoch.set_postfix(loss=loss.item())

    print(f"Epoch: {e} - loss: {e_loss/len(train_data)}")

Start training on cuda [...]


  0%|          | 0/391 [00:00<?, ?batch/s]

Epoch: 0 - loss: 0.059229094747304915


  0%|          | 0/391 [00:00<?, ?batch/s]

Epoch: 1 - loss: 0.01597552524626255


In [7]:
## A simple evaluation step.
t_loss = 0
correct = 0

model.eval()
with torch.no_grad():
    for i, data in (tepoch := tqdm(enumerate(test_loader), unit="batch", total=len(test_loader))):
        tepoch.set_description("Evaluation")
        x, y = data[0].to(DEVICE), data[1].to(DEVICE)

        # This get's the prediction from the network.
        output = model(x)
        # Sum up batch loss.
        t_loss += criterion(output, y.reshape(-1, 1).float()).item()

        # Get the index of the max log-probability.
        output = (torch.sigmoid(output) > 0.5).int()
        correct += output.eq(y.view_as(output)).sum().item()

t_loss /= len(test_loader.dataset)

print('AVG loss: {:.4f}, ACC: {}/{} ({:.0f}%)'.format(
      t_loss, correct, len(test_loader.dataset),
      100. * correct / len(test_loader.dataset)))

  0%|          | 0/194 [00:00<?, ?batch/s]

AVG loss: 0.0005, ACC: 12280/12461 (99%)
